
# AgroUTMACH PIX
## NB04B — Corrección visual del tracking: mostrar claramente aves en movimiento

### ¿Por qué se veían pocos pollos resaltados?
En NB04 se resaltaron únicamente **25 tracks persistentes** para evitar saturar la pantalla.
Eso NO significa que solo esos pollos estuvieran siendo seguidos.

En el video de NB04 podían existir más de 300 tracks activos al mismo tiempo,
pero la mayoría se dibujaban con cajas grises discretas.

### Esta revisión cambia la lógica visual:
- **Todas las aves con track activo** tendrán caja visible.
- Las aves con **movimiento relativo alto** se resaltarán en amarillo.
- Las aves con movimiento bajo se mostrarán en cian.
- Solo una fracción de las aves en movimiento dejará una trayectoria corta para no llenar la pantalla de líneas.
- El panel mostrará:
  - aves visibles;
  - tracks activos;
  - aves con movimiento relativo alto.

> El movimiento se estima en píxeles/segundo dentro de la imagen. No representa velocidad física, estrés ni enfermedad.

**Este notebook usa CPU. No vuelve a ejecutar YOLO ni BoT-SORT.**


In [ ]:

from google.colab import drive
drive.mount('/content/drive')


In [ ]:

from pathlib import Path
from collections import defaultdict
import os, shutil, math
import cv2
import numpy as np
import pandas as pd

ROOT = Path("/content/drive/MyDrive/AgroUTMACH_PIX")

TRACK_CSV = ROOT / "05_RESULTADOS" / "NB04_TRACKING" / "NB04_tracks_por_frame.csv"
VIDEO_DRIVE = ROOT / "04_VIDEOS_FERIA" / "PIO_tracking_estable_original_20s.mp4"

VIDEOS_OUT = ROOT / "04_VIDEOS_FERIA"
DASHBOARD_ASSETS = ROOT / "06_DASHBOARD" / "assets"
RESULTS_OUT = ROOT / "05_RESULTADOS" / "NB04B_MOVIMIENTO_VISUAL"

RESULTS_OUT.mkdir(parents=True, exist_ok=True)
DASHBOARD_ASSETS.mkdir(parents=True, exist_ok=True)

assert TRACK_CSV.exists(), f"Falta: {TRACK_CSV}"
assert VIDEO_DRIVE.exists(), f"Falta: {VIDEO_DRIVE}"

tracks = pd.read_csv(TRACK_CSV)
print("Registros:", f"{len(tracks):,}")
print("IDs temporales:", tracks["track_id"].nunique())



## 1. Preparar centroides y movimiento relativo

Para reducir el efecto de pequeñas vibraciones de la caja,
el movimiento se calcula comparando posiciones separadas aproximadamente **0,5 segundos**.

El umbral se obtiene automáticamente a partir de la distribución de velocidades.
Se resaltará aproximadamente el grupo superior de movimiento relativo.


In [ ]:

VIDEO_LOCAL = Path("/content/PIO_tracking_estable_original_20s.mp4")
if not VIDEO_LOCAL.exists() or VIDEO_LOCAL.stat().st_size != VIDEO_DRIVE.stat().st_size:
    shutil.copy2(VIDEO_DRIVE, VIDEO_LOCAL)

cap = cv2.VideoCapture(str(VIDEO_LOCAL))
assert cap.isOpened(), "No se pudo abrir el video."

fps = cap.get(cv2.CAP_PROP_FPS)
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

tracks["cx"] = (tracks["x1"] + tracks["x2"]) / 2
tracks["cy"] = (tracks["y1"] + tracks["y2"]) / 2

LOOKBACK = max(1, int(round(0.5 * fps)))  # ~0.5 s

# Crear tabla por track y frame para buscar una posición previa.
pos_lookup = {
    (int(r.track_id), int(r.frame)): (float(r.cx), float(r.cy))
    for r in tracks.itertuples()
}

speeds = []
speed_values = []

for r in tracks.itertuples():
    tid = int(r.track_id)
    fr = int(r.frame)
    prev_fr = fr - LOOKBACK

    prev = pos_lookup.get((tid, prev_fr))
    if prev is None:
        speeds.append(np.nan)
        continue

    dx = float(r.cx) - prev[0]
    dy = float(r.cy) - prev[1]
    dist = math.hypot(dx, dy)
    speed = dist / (LOOKBACK / fps)  # px/s

    speeds.append(speed)
    speed_values.append(speed)

tracks["speed_px_s_05s"] = speeds

valid_speeds = np.array(speed_values, dtype=float)
assert len(valid_speeds) > 0, "No se pudieron calcular velocidades."

# Umbral dinámico: percentil 60 -> aproximadamente 40% con mayor movimiento.
MOVE_THRESHOLD = float(np.percentile(valid_speeds, 60))

print(f"FPS: {fps:.1f}")
print(f"Lookback: {LOOKBACK} frames (~{LOOKBACK/fps:.2f} s)")
print(f"Mediana movimiento: {np.median(valid_speeds):.2f} px/s")
print(f"Umbral P60: {MOVE_THRESHOLD:.2f} px/s")
print(f"P90: {np.percentile(valid_speeds,90):.2f} px/s")

tracks.to_csv(RESULTS_OUT / "NB04B_tracks_con_movimiento.csv", index=False)



## 2. Renderizar video corregido

### Código visual
- **Amarillo:** movimiento relativo alto.
- **Cian:** track activo con movimiento relativo bajo o sin suficiente historia.
- **Trayectoria amarilla:** se muestra solo para un subconjunto de aves en movimiento para mantener legibilidad.

Así la audiencia verá claramente que el sistema está siguiendo muchas aves simultáneamente.


In [ ]:

FINAL_TEMP = Path("/content/PIO_tracking_movimiento_visual_temp.mp4")
FINAL_OUT = VIDEOS_OUT / "PIO_tracking_movimiento_visual_20s.mp4"

cap = cv2.VideoCapture(str(VIDEO_LOCAL))
writer = cv2.VideoWriter(
    str(FINAL_TEMP),
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

by_frame = {int(k): g for k, g in tracks.groupby("frame")}
history = defaultdict(list)

# Historial corto (~1.2 s) para trayectorias
MAX_HISTORY = max(8, int(round(1.2 * fps)))

# Para no dibujar 100+ líneas, dejamos trayectoria en un subconjunto estable
# de IDs en movimiento. Las cajas de TODOS sí se muestran.
TRAIL_MOD = 3

frame_idx = 0

while True:
    ok, frame = cap.read()
    if not ok:
        break

    current = by_frame.get(frame_idx)
    moving_count = 0
    active_count = 0

    if current is not None and len(current) > 0:
        active_count = int(current["track_id"].nunique())

        for _, r in current.iterrows():
            tid = int(r["track_id"])
            x1, y1, x2, y2 = map(int, [r["x1"], r["y1"], r["x2"], r["y2"]])
            cx = int(r["cx"])
            cy = int(r["cy"])

            speed = r["speed_px_s_05s"]
            is_moving = pd.notna(speed) and float(speed) >= MOVE_THRESHOLD

            if is_moving:
                moving_count += 1
                color = (0, 255, 255)   # amarillo BGR
                thickness = 2
            else:
                color = (255, 255, 0)   # cian BGR
                thickness = 1

            # Caja para TODOS los tracks
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)

            # ID pequeño para aves en movimiento
            if is_moving:
                cv2.putText(
                    frame, f"{tid}",
                    (x1, max(16, y1 - 4)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.34,
                    color, 1, cv2.LINE_AA
                )

            # Historial de centroides
            history[tid].append((cx, cy))
            if len(history[tid]) > MAX_HISTORY:
                history[tid] = history[tid][-MAX_HISTORY:]

            # Trayectoria solo en ~1/3 de las aves en movimiento
            if is_moving and tid % TRAIL_MOD == 0 and len(history[tid]) > 1:
                pts = history[tid]
                for p1, p2 in zip(pts[:-1], pts[1:]):
                    cv2.line(frame, p1, p2, (0, 255, 255), 2, cv2.LINE_AA)

    # Panel
    overlay = frame.copy()
    cv2.rectangle(overlay, (20, 18), (760, 170), (0, 0, 0), -1)
    frame = cv2.addWeighted(overlay, 0.56, frame, 0.44, 0)

    cv2.putText(
        frame, "AgroUTMACH PIX | Tracking y movimiento",
        (38, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.90,
        (255,255,255), 2, cv2.LINE_AA
    )
    cv2.putText(
        frame, f"Aves visibles / tracks activos: {active_count}",
        (38, 93), cv2.FONT_HERSHEY_SIMPLEX, 0.72,
        (255,255,255), 2, cv2.LINE_AA
    )
    cv2.putText(
        frame, f"Movimiento relativo alto: {moving_count}",
        (38, 129), cv2.FONT_HERSHEY_SIMPLEX, 0.72,
        (0,255,255), 2, cv2.LINE_AA
    )
    cv2.putText(
        frame, "Amarillo = mayor movimiento | Cian = menor movimiento",
        (38, 158), cv2.FONT_HERSHEY_SIMPLEX, 0.48,
        (255,255,255), 1, cv2.LINE_AA
    )

    writer.write(frame)
    frame_idx += 1

cap.release()
writer.release()

cmd = (
    f'ffmpeg -y -hide_banner -loglevel error '
    f'-i "{FINAL_TEMP}" -c:v libx264 -preset fast -crf 20 '
    f'-pix_fmt yuv420p -an "{FINAL_OUT}"'
)
rc = os.system(cmd)
assert rc == 0 and FINAL_OUT.exists(), "Falló la codificación final."

print("✓ Video corregido:", FINAL_OUT)



## 3. Actualizar automáticamente el dashboard

Esta celda reemplaza **solo el video de tracking** dentro del paquete del dashboard.
No modifica los demás videos ni los resultados anteriores.


In [ ]:

DASHBOARD_TRACK_VIDEO = DASHBOARD_ASSETS / "03_tracking_trayectorias_20s.mp4"

shutil.copy2(FINAL_OUT, DASHBOARD_TRACK_VIDEO)

print("✓ Dashboard actualizado:")
print(DASHBOARD_TRACK_VIDEO)
print()
print("IMPORTANTE:")
print("En su PC reemplace el archivo local")
print("assets/03_tracking_trayectorias_20s.mp4")
print("por esta versión actualizada del Drive.")



## 4. Interpretación para la feria

Cuando un visitante pregunte por los colores:

> **Cian:** ave detectada y seguida con menor desplazamiento relativo durante el intervalo reciente.  
> **Amarillo:** ave detectada y seguida con mayor desplazamiento relativo.  
> **Línea amarilla:** trayectoria reciente de una muestra de tracks en movimiento.

No afirmar:
- identidad individual permanente;
- velocidad física en m/s;
- estrés;
- enfermedad;
- bienestar animal.


In [ ]:

print("=" * 76)
print("NB04B FINALIZADO CORRECTAMENTE")
print(f"Umbral movimiento relativo: {MOVE_THRESHOLD:.2f} px/s")
print("Video:", FINAL_OUT)
print("Dashboard: video de tracking actualizado")
print("=" * 76)
